In [0]:
%sql
select * from business_to_business.gold.fact_orders;

date,product_code,customer_code,sold_quantity
2024-01-01,ARCHDDE20D,70002017,161
2024-01-01,ARCH158F41,70002017,133
2024-01-01,ARCHAFF0E4,70002017,76
2024-01-01,ARCH6B94F7,70002017,92
2024-01-01,ARCH5D1FE7,70002017,117
2024-01-01,ARCH7B49A9,70002017,36
2024-01-01,ARCH497D34,70002017,98
2024-01-01,ARCHE71D79,70002017,156
2024-01-01,BADM88C384,70002017,28
2024-01-01,BADMA5EBA3,70002017,33


In [0]:
min_date = spark.sql("""
                        SELECT min(date) as min_date
                        FROM 
                        business_to_business.gold.fact_orders
                        """)
min_date = min_date.collect()[0][0]

In [0]:
max_date = spark.sql("""
                        SELECT max(date) as min_date
                        FROM 
                        business_to_business.gold.fact_orders
                        """)
max_date = max_date.collect()[0][0]
print(max_date)

2025-11-01


In [0]:
months_df = (
    spark.sql(f"""
              select explode(
                  sequence(DATE'{min_date}', DATE'{max_date}', interval 1 month) ) as month_start_date
              
              """)
)
# print(months_division.show())
display(months_df)

month_start_date
2024-01-01
2024-02-01
2024-03-01
2024-04-01
2024-05-01
2024-06-01
2024-07-01
2024-08-01
2024-09-01
2024-10-01


In [0]:
#Now i want to perform the dim_date_table
from pyspark.sql import functions as F
from datetime import datetime

df = (
    months_df
    .withColumn("date_key", F.date_format("month_start_date", "yyyyMM").cast("int"))
    .withColumn("year", F.year("month_start_date"))
    .withColumn("month_name", F.date_format("month_start_date", "MMMM"))
    .withColumn("month_short_name", F.date_format("month_start_date", "MMM"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("month_start_date")))
    .withColumn("year_quarter", F.concat(F.col("year"), F.lit("-Q"), F.quarter("month_start_date")))
)


In [0]:
display(df)

month_start_date,date_key,year,month_name,month_short_name,quarter,year_quarter
2024-01-01,202401,2024,January,Jan,Q1,2024-Q1
2024-02-01,202402,2024,February,Feb,Q1,2024-Q1
2024-03-01,202403,2024,March,Mar,Q1,2024-Q1
2024-04-01,202404,2024,April,Apr,Q2,2024-Q2
2024-05-01,202405,2024,May,May,Q2,2024-Q2
2024-06-01,202406,2024,June,Jun,Q2,2024-Q2
2024-07-01,202407,2024,July,Jul,Q3,2024-Q3
2024-08-01,202408,2024,August,Aug,Q3,2024-Q3
2024-09-01,202409,2024,September,Sep,Q3,2024-Q3
2024-10-01,202410,2024,October,Oct,Q4,2024-Q4


In [0]:
display(df)

month_start_date,date_key,year,month_name,month_short_name,quarter,year_quarter
2024-01-01,202401,2024,January,Jan,Q1,2024-Q1
2024-02-01,202402,2024,February,Feb,Q1,2024-Q1
2024-03-01,202403,2024,March,Mar,Q1,2024-Q1
2024-04-01,202404,2024,April,Apr,Q2,2024-Q2
2024-05-01,202405,2024,May,May,Q2,2024-Q2
2024-06-01,202406,2024,June,Jun,Q2,2024-Q2
2024-07-01,202407,2024,July,Jul,Q3,2024-Q3
2024-08-01,202408,2024,August,Aug,Q3,2024-Q3
2024-09-01,202409,2024,September,Sep,Q3,2024-Q3
2024-10-01,202410,2024,October,Oct,Q4,2024-Q4


In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("business_to_business.gold.dim_date")
)